# M4.2 | Statistics, transformation and feature selection

## Using skew, kurtosis, Q–Q plots, normality tests and ANOVA in a machine learning pipeline

**Teaching time:** 3 hours  
**Breaks:** Two 10-minute breaks  
**Level:** Beginner  
**Machine learning task:** Predict learner engagement score

This notebook continues the university-performance case study.

It uses:

- `university_performance(4).csv`
- `university_support_lookup(4).csv`

Place both files in the same folder as this notebook.

## The practical decision pathway

We will follow this workflow:

1. Load, clean and join the data.
2. Use skew and kurtosis to identify unusual distribution shapes.
3. Use Q–Q plots, Shapiro–Wilk and K–S as normality diagnostics.
4. Test whether a transformation improves the feature representation.
5. Use one-way ANOVA to screen a categorical feature.
6. Use an ANOVA F-test to select processed features.
7. Compare the complete pipelines on unseen data.

> Normality diagnostics inform transformation decisions.
>
> ANOVA and F-tests inform feature relevance.
>
> Test performance informs the final modelling decision.

## Learning outcomes

By the end of this notebook, you will be able to:

1. load and join two related CSV files
2. distinguish normality diagnostics from feature-selection tests
3. interpret skew and excess kurtosis
4. inspect a feature using a Q–Q plot
5. run Shapiro–Wilk and K–S normality diagnostics
6. test a Yeo–Johnson transformation
7. run a practical one-way ANOVA test
8. explain how ANOVA supports feature screening
9. use `f_regression` inside a feature-selection pipeline
10. compare complete pipelines using unseen test data

## Three-hour learning plan

| Time | Duration | Section |
|---|---:|---|
| 00:00–00:10 | 10 min | Purpose and decision pathway |
| 00:10–00:30 | 20 min | Load, clean and join the datasets |
| 00:30–00:45 | 15 min | Define the target and split the data |
| 00:45–01:05 | 20 min | Skew, kurtosis and distribution shape |
| 01:05–01:20 | 15 min | Q–Q plots, Shapiro–Wilk and K–S |
| 01:20–01:30 | 10 min | **Break 1** |
| 01:30–01:55 | 25 min | Test a Yeo–Johnson transformation |
| 01:55–02:15 | 20 min | Practical one-way ANOVA |
| 02:15–02:30 | 15 min | Build and compare preprocessing pipelines |
| 02:30–02:40 | 10 min | **Break 2** |
| 02:40–02:55 | 15 min | Add F-test feature selection |
| 02:55–03:00 | 5 min | Recommendation and exit check |

<details>
<summary><strong>Facilitator guidance</strong></summary>

Keep returning to three questions:

1. **Shape:** Does the feature need investigation or transformation?
2. **Relevance:** Does the feature contain information about the target?
3. **Performance:** Does the complete pipeline work better on unseen data?

Avoid suggesting that every machine learning input must be normally distributed.

Shapiro–Wilk, K–S and Q–Q plots are diagnostic tools. They are not automatic feature-removal rules.

</details>

# Part 1 | Load and prepare the data

## Step 1: Import the tools

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectPercentile, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, StandardScaler

RANDOM_STATE = 42
ALPHA = 0.05

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

print('Setup complete.')

## Step 2: Load the existing datasets

The notebook expects both CSV files to be in the same folder as the notebook.

In [ ]:
PERFORMANCE_FILE = Path('university_performance(4).csv')
SUPPORT_FILE = Path('university_support_lookup(4).csv')

for file_path in [PERFORMANCE_FILE, SUPPORT_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f'Could not find {file_path.name}. '
            'Place it in the same folder as this notebook.'
        )

performance_raw = pd.read_csv(PERFORMANCE_FILE)
support_raw = pd.read_csv(SUPPORT_FILE)

print('Performance dataset:', performance_raw.shape)
print('Support lookup:', support_raw.shape)

display(performance_raw.head())
display(support_raw.head())

## Step 3: Standardise column names and text values

Consistent labels make the join and later feature-selection steps more reliable.

In [ ]:
def standardise_column_names(dataframe):
    cleaned = dataframe.copy()

    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^a-z0-9]+', '_', regex=True)
        .str.strip('_')
    )

    return cleaned


performance = standardise_column_names(performance_raw)
support = standardise_column_names(support_raw)

for dataframe in [performance, support]:
    for column in ['region', 'programme']:
        if column in dataframe.columns:
            dataframe[column] = (
                dataframe[column]
                .astype('string')
                .str.strip()
                .str.title()
            )

print('Performance columns:')
print(performance.columns.tolist())

print('\nSupport columns:')
print(support.columns.tolist())

## Step 4: Convert known measures to numbers

CSV files sometimes store percentages and scores as text.

In [ ]:
known_measure_columns = [
    'attendance_rate',
    'engagement_score',
    'submission_rate',
    'target_engagement_score',
    'target_attendance_rate',
]


def convert_measure_to_numeric(series):
    cleaned = (
        series
        .astype('string')
        .str.strip()
        .str.replace('%', '', regex=False)
        .str.replace(',', '', regex=False)
    )

    return pd.to_numeric(cleaned, errors='coerce')


for dataframe in [performance, support]:
    for column in known_measure_columns:
        if column in dataframe.columns:
            dataframe[column] = convert_measure_to_numeric(
                dataframe[column]
            )

print('Known numerical measures converted.')

## Step 5: Join the support lookup

A left join keeps every university-performance row and adds matching regional support information.

In [ ]:
if 'region' not in performance.columns:
    raise KeyError('The performance dataset does not contain region.')

if 'region' not in support.columns:
    raise KeyError('The support lookup does not contain region.')

if support['region'].duplicated().any():
    duplicate_regions = (
        support.loc[
            support['region'].duplicated(keep=False),
            'region',
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        f'The support lookup contains duplicate regions: {duplicate_regions}'
    )

university_data = performance.merge(
    support,
    on='region',
    how='left',
    validate='many_to_one',
    indicator='lookup_match',
    suffixes=('_performance', '_support'),
)

display(
    university_data['lookup_match']
    .value_counts(dropna=False)
    .to_frame('rows')
)

university_data = university_data.drop(columns='lookup_match')

print('Joined dataset:', university_data.shape)
display(university_data.head())

# Part 2 | Define the machine learning question

## Step 6: Select the target

The practical question is:

> Can university, programme, participation and support features help predict `engagement_score`?

This is a regression problem because engagement score is numerical.

In [ ]:
TARGET_COLUMN = 'engagement_score'

if TARGET_COLUMN not in university_data.columns:
    raise KeyError(
        f'{TARGET_COLUMN!r} was not found. '
        f'Available columns: {university_data.columns.tolist()}'
    )

university_data[TARGET_COLUMN] = pd.to_numeric(
    university_data[TARGET_COLUMN],
    errors='coerce',
)

model_data = university_data.dropna(
    subset=[TARGET_COLUMN]
).copy()

print('Rows available for modelling:', len(model_data))
display(model_data[TARGET_COLUMN].describe().to_frame())

## Step 7: Create training and test data

All transformation and feature-selection decisions must learn from the training data.

The test data represents unseen information.

In [ ]:
X_raw = model_data.drop(columns=[TARGET_COLUMN])
y = model_data[TARGET_COLUMN].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print('Training features:', X_train_raw.shape)
print('Test features:', X_test_raw.shape)

## Step 8: Remove unsuitable modelling columns

We remove:

- identifiers
- constant columns
- direct engagement benchmarks or derived engagement fields
- very high-cardinality text fields

These decisions are based on the training data.

In [ ]:
identifier_columns = [
    column
    for column in X_train_raw.columns
    if (
        column == 'id'
        or column.endswith('_id')
        or column in ['student_name', 'learner_name']
    )
]

target_proxy_columns = [
    column
    for column in [
        'target_engagement_score',
        'engagement_gap',
        'engagement_difference',
        'engagement_band',
    ]
    if column in X_train_raw.columns
]

constant_columns = [
    column
    for column in X_train_raw.columns
    if X_train_raw[column].nunique(dropna=False) <= 1
]

text_columns = (
    X_train_raw
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

high_cardinality_columns = [
    column
    for column in text_columns
    if (
        X_train_raw[column].nunique(dropna=True) > 50
        and X_train_raw[column].nunique(dropna=True) / len(X_train_raw) > 0.20
    )
]

columns_to_remove = sorted(
    set(
        identifier_columns
        + target_proxy_columns
        + constant_columns
        + high_cardinality_columns
    )
)

X_train = X_train_raw.drop(
    columns=columns_to_remove,
    errors='ignore',
)

X_test = X_test_raw.drop(
    columns=columns_to_remove,
    errors='ignore',
)

print('Columns removed:')
print(columns_to_remove if columns_to_remove else 'None')

print('\nFeatures remaining:', X_train.shape[1])

# Part 3 | Investigate distribution shape

## Skew and kurtosis

**Skew** describes asymmetry.

**Excess kurtosis** helps describe tail weight and extreme observations.

| Measure | Working interpretation |
|---|---|
| Absolute skew below 0.5 | Roughly symmetric |
| Absolute skew from 0.5 to below 1 | Moderately skewed |
| Absolute skew of 1 or more | Strongly skewed: investigate |
| Excess kurtosis between -1 and 1 | Closer to normal-like tail weight |
| Excess kurtosis above 1 | Heavier tails |
| Excess kurtosis well above 3 | Inspect extreme observations |

These thresholds support investigation. They are not automatic removal rules.

In [ ]:
numerical_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

shape_summary = pd.DataFrame(
    {
        'missing_values': X_train[numerical_features].isna().sum(),
        'unique_values': X_train[numerical_features].nunique(),
        'mean': X_train[numerical_features].mean(),
        'median': X_train[numerical_features].median(),
        'skew': X_train[numerical_features].skew(),
        'excess_kurtosis': X_train[numerical_features].kurt(),
    }
)

shape_summary['absolute_skew'] = shape_summary['skew'].abs()

shape_summary = shape_summary.sort_values(
    ['absolute_skew', 'excess_kurtosis'],
    ascending=[False, False],
)

display(shape_summary)

## Step 9: Plot the most skewed features

The notebook automatically selects up to three numerical features with the highest absolute skew.

In [ ]:
priority_features = (
    shape_summary
    .dropna(subset=['absolute_skew'])
    .query('unique_values > 2')
    .head(3)
    .index
    .tolist()
)

print('Features selected for detailed diagnostics:')
print(priority_features)

for feature in priority_features:
    values = X_train[feature].dropna()

    plt.figure(figsize=(8, 3.5))
    plt.hist(values, bins=25)
    plt.axvline(values.mean(), linestyle='--', label='Mean')
    plt.axvline(values.median(), linestyle=':', label='Median')
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Number of records')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Part 4 | Use Q–Q plots and normality tests

## Q–Q plots

A Q–Q plot compares the observed values with values expected from a normal distribution.

Look for:

- points close to the line: closer agreement with normality
- a curved pattern: skew or another systematic difference
- departures at the ends: unusually light or heavy tails
- isolated points: possible extreme observations

A Q–Q plot is a visual diagnostic. It does not measure predictive usefulness.

In [ ]:
for feature in priority_features:
    values = X_train[feature].dropna()

    plt.figure(figsize=(6, 5))
    stats.probplot(
        values,
        dist='norm',
        plot=plt,
    )
    plt.title(f'Q–Q plot: {feature}')
    plt.tight_layout()
    plt.show()

## Shapiro–Wilk and K–S diagnostics

The working null hypothesis is:

> The tested values are consistent with a normal distribution.

Using `alpha = 0.05`:

- `p < 0.05`: evidence against normality
- `p >= 0.05`: insufficient evidence to reject normality

A non-significant result does not prove that a feature is normal.

The K–S result below is an approximate teaching diagnostic because the mean and standard deviation are estimated from the same sample.

In [ ]:
def normality_diagnostics(
    dataframe,
    features,
    maximum_sample_size=5000,
):
    results = []

    for feature in features:
        values = dataframe[feature].dropna()

        if len(values) < 3 or values.nunique() < 3:
            continue

        if len(values) > maximum_sample_size:
            tested_values = values.sample(
                maximum_sample_size,
                random_state=RANDOM_STATE,
            )
        else:
            tested_values = values

        shapiro_statistic, shapiro_p_value = stats.shapiro(
            tested_values
        )

        standard_deviation = tested_values.std(ddof=1)

        if standard_deviation > 0:
            standardised_values = (
                tested_values - tested_values.mean()
            ) / standard_deviation

            ks_statistic, ks_p_value = stats.kstest(
                standardised_values,
                'norm',
            )
        else:
            ks_statistic = np.nan
            ks_p_value = np.nan

        results.append(
            {
                'feature': feature,
                'n_tested': len(tested_values),
                'shapiro_statistic': shapiro_statistic,
                'shapiro_p_value': shapiro_p_value,
                'ks_statistic': ks_statistic,
                'ks_p_value_approximate': ks_p_value,
            }
        )

    return pd.DataFrame(results).set_index('feature')


normality_results = normality_diagnostics(
    X_train,
    priority_features,
)

normality_results = normality_results.join(
    shape_summary[['skew', 'excess_kurtosis']],
    how='left',
)

display(normality_results)

## Combine the evidence

| Evidence | Practical response |
|---|---|
| Straight Q–Q plot and low skew | Leave unchanged initially |
| Curved Q–Q plot and strong skew | Test a transformation |
| Tail departures and high kurtosis | Investigate unusual observations |
| Significant test but small visual departure | Consider whether the difference is practically important |
| Non-significant test but visible departure | The sample may be too small to detect it |

Normality diagnostics help decide what to test next. They do not decide whether a feature should be selected.

# Break 1 | 10 minutes

You have completed:

- loading and joining the datasets
- defining the target
- skew and kurtosis
- histograms
- Q–Q plots
- Shapiro–Wilk
- K–S diagnostics

When you return, you will test a transformation.

# Part 5 | Test a Yeo–Johnson transformation

Yeo–Johnson can work with:

- positive values
- zero
- negative values

We will apply it to non-binary numerical features with strong skew.

In [ ]:
SKEW_THRESHOLD = 1.0

binary_numeric_features = [
    feature
    for feature in numerical_features
    if X_train[feature].nunique(dropna=True) <= 2
]

skewed_numeric_features = [
    feature
    for feature in numerical_features
    if (
        feature not in binary_numeric_features
        and pd.notna(shape_summary.loc[feature, 'skew'])
        and abs(shape_summary.loc[feature, 'skew']) >= SKEW_THRESHOLD
    )
]

regular_numeric_features = [
    feature
    for feature in numerical_features
    if feature not in skewed_numeric_features
]

print('Features shortlisted for transformation:')
print(
    skewed_numeric_features
    if skewed_numeric_features
    else 'No feature reached the skew threshold.'
)

## Preview the transformation

This exploratory preview uses the training data only.

The final transformer will be fitted again inside the pipeline.

In [ ]:
if skewed_numeric_features:
    preview_imputer = SimpleImputer(strategy='median')

    preview_before_array = preview_imputer.fit_transform(
        X_train[skewed_numeric_features]
    )

    preview_before = pd.DataFrame(
        preview_before_array,
        columns=skewed_numeric_features,
        index=X_train.index,
    )

    preview_transformer = PowerTransformer(
        method='yeo-johnson',
        standardize=False,
    )

    preview_after_array = preview_transformer.fit_transform(
        preview_before
    )

    preview_after = pd.DataFrame(
        preview_after_array,
        columns=skewed_numeric_features,
        index=X_train.index,
    )

    transformation_comparison = pd.DataFrame(
        {
            'before_skew': preview_before.skew(),
            'after_skew': preview_after.skew(),
            'before_excess_kurtosis': preview_before.kurt(),
            'after_excess_kurtosis': preview_after.kurt(),
            'learned_lambda': preview_transformer.lambdas_,
        }
    )

    display(transformation_comparison)
else:
    print('There are no shortlisted features to transform.')

## Compare Q–Q plots before and after transformation

In [ ]:
if skewed_numeric_features:
    example_feature = skewed_numeric_features[0]

    plt.figure(figsize=(6, 5))
    stats.probplot(
        preview_before[example_feature],
        dist='norm',
        plot=plt,
    )
    plt.title(f'Before transformation: {example_feature}')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    stats.probplot(
        preview_after[example_feature],
        dist='norm',
        plot=plt,
    )
    plt.title(f'After Yeo–Johnson: {example_feature}')
    plt.tight_layout()
    plt.show()
else:
    print('There is no transformed feature to plot.')

## Repeat the normality diagnostics

The aim is not to achieve perfect normality.

We are checking whether the transformation produces a more useful representation.

In [ ]:
if skewed_numeric_features:
    features_to_compare = skewed_numeric_features[:3]

    before_tests = normality_diagnostics(
        preview_before,
        features_to_compare,
    )

    after_tests = normality_diagnostics(
        preview_after,
        features_to_compare,
    )

    test_comparison = (
        before_tests[
            ['shapiro_p_value', 'ks_p_value_approximate']
        ]
        .add_prefix('before_')
        .join(
            after_tests[
                ['shapiro_p_value', 'ks_p_value_approximate']
            ].add_prefix('after_')
        )
    )

    display(test_comparison)
else:
    print('There are no transformed features to compare.')

# Part 6 | Use one-way ANOVA as a practical screening tool

One-way ANOVA helps investigate whether a categorical feature is associated with a numerical target.

For example:

> Is the average engagement score different across programmes?

### Hypotheses

- **Null hypothesis:** all group mean engagement scores are equal.
- **Alternative hypothesis:** at least one group mean is different.

Using `alpha = 0.05`:

- `p < 0.05`: retain the feature as a promising candidate
- `p >= 0.05`: treat it as a weaker candidate

ANOVA is a screening tool. The final decision still depends on model performance.

### Optional stretch: Create a reusable ANOVA function

The function keeps the implementation short:

1. group the numerical target by a categorical feature
2. run `stats.f_oneway()`
3. return the F-statistic and p-value

In [ ]:
def run_one_way_anova(
    dataframe,
    group_column,
    target_column,
    minimum_group_size=5,
):
    clean_data = dataframe[
        [group_column, target_column]
    ].dropna()

    groups = [
        group[target_column].to_numpy()
        for _, group in clean_data.groupby(group_column)
        if len(group) >= minimum_group_size
    ]

    if len(groups) < 2:
        raise ValueError(
            'ANOVA requires at least two groups with enough records.'
        )

    f_statistic, p_value = stats.f_oneway(*groups)

    return f_statistic, p_value

## Step 11: Run the ANOVA test

The notebook uses `programme` when available. Otherwise, it uses `region`.

In [ ]:
if 'programme' in X_train_raw.columns:
    GROUP_COLUMN = 'programme'
elif 'region' in X_train_raw.columns:
    GROUP_COLUMN = 'region'
else:
    GROUP_COLUMN = None


if GROUP_COLUMN is not None:
    anova_data = pd.DataFrame(
        {
            GROUP_COLUMN: X_train_raw.loc[y_train.index, GROUP_COLUMN],
            TARGET_COLUMN: y_train,
        }
    )

    f_statistic, p_value = run_one_way_anova(
        anova_data,
        group_column=GROUP_COLUMN,
        target_column=TARGET_COLUMN,
    )

    print('Feature tested:', GROUP_COLUMN)
    print('F-statistic:', round(f_statistic, 4))
    print('p-value:', round(p_value, 4))

    if p_value < ALPHA:
        print(
            f'{GROUP_COLUMN} shows evidence of different mean '
            f'{TARGET_COLUMN} values.'
        )
        print(
            'Action: retain it as a candidate and test it in the pipeline.'
        )
    else:
        print(
            f'There is insufficient evidence that mean {TARGET_COLUMN} '
            f'differs across {GROUP_COLUMN}.'
        )
        print(
            'Action: treat it as a weaker candidate and check model performance.'
        )
else:
    print('No suitable categorical feature was found for ANOVA.')

## How ANOVA links to feature selection

The practical workflow is:

1. Run ANOVA to screen a categorical feature.
2. Retain promising features as candidates.
3. Encode categories inside the preprocessing pipeline.
4. Run automated feature selection after preprocessing.
5. Compare model performance on unseen data.

> A significant ANOVA result means the feature is worth investigating.
>
> It does not mean the feature must be selected.

# Part 7 | Build leakage-safe pipelines

The baseline pipeline imputes and scales numerical features.

The transformed pipeline applies Yeo–Johnson to the strongly skewed numerical features.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown='ignore',
            sparse=False,
        )


def make_baseline_preprocessor():
    transformers = []

    if numerical_features:
        numerical_pipeline = Pipeline(
            steps=[
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]
        )

        transformers.append(
            ('numeric', numerical_pipeline, numerical_features)
        )

    if categorical_features:
        categorical_pipeline = Pipeline(
            steps=[
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('encode', make_one_hot_encoder()),
            ]
        )

        transformers.append(
            ('categorical', categorical_pipeline, categorical_features)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder='drop',
        verbose_feature_names_out=True,
    )


def make_transformed_preprocessor():
    transformers = []

    if skewed_numeric_features:
        skewed_pipeline = Pipeline(
            steps=[
                ('impute', SimpleImputer(strategy='median')),
                (
                    'power',
                    PowerTransformer(
                        method='yeo-johnson',
                        standardize=True,
                    ),
                ),
            ]
        )

        transformers.append(
            (
                'skewed_numeric',
                skewed_pipeline,
                skewed_numeric_features,
            )
        )

    if regular_numeric_features:
        regular_pipeline = Pipeline(
            steps=[
                ('impute', SimpleImputer(strategy='median')),
                ('scale', StandardScaler()),
            ]
        )

        transformers.append(
            (
                'regular_numeric',
                regular_pipeline,
                regular_numeric_features,
            )
        )

    if categorical_features:
        categorical_pipeline = Pipeline(
            steps=[
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('encode', make_one_hot_encoder()),
            ]
        )

        transformers.append(
            ('categorical', categorical_pipeline, categorical_features)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder='drop',
        verbose_feature_names_out=True,
    )


print('Preprocessing functions ready.')

## Step 12: Create the evaluation function

We will compare:

- mean absolute error
- root mean squared error
- R-squared

Lower error is better. Higher R-squared is better.

In [ ]:
def evaluate_pipeline(pipeline_name, pipeline):
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    return {
        'pipeline': pipeline_name,
        'mean_absolute_error': mean_absolute_error(
            y_test,
            predictions,
        ),
        'root_mean_squared_error': np.sqrt(
            mean_squared_error(
                y_test,
                predictions,
            )
        ),
        'r_squared': r2_score(
            y_test,
            predictions,
        ),
    }

In [ ]:
baseline_pipeline = Pipeline(
    steps=[
        ('preprocess', make_baseline_preprocessor()),
        ('model', Ridge(alpha=1.0)),
    ]
)

transformed_pipeline = Pipeline(
    steps=[
        ('preprocess', make_transformed_preprocessor()),
        ('model', Ridge(alpha=1.0)),
    ]
)

baseline_result = evaluate_pipeline(
    'Baseline: imputation and scaling',
    baseline_pipeline,
)

transformed_result = evaluate_pipeline(
    'Yeo–Johnson transformation',
    transformed_pipeline,
)

initial_results = [
    baseline_result,
    transformed_result,
]

display(
    pd.DataFrame(initial_results)
    .set_index('pipeline')
    .round(4)
)

### Interpret the comparison

Ask:

1. Did the transformed distributions look closer to normal?
2. Did the transformed pipeline reduce prediction error?
3. Did R-squared improve?
4. Is the difference large enough to justify the added complexity?

Improved normality and improved prediction are different outcomes.

# Break 2 | 10 minutes

You have completed:

- normality diagnostics
- transformation testing
- one-way ANOVA
- baseline preprocessing
- transformed preprocessing
- initial model comparison

When you return, you will add automated feature selection.

# Part 8 | Use an ANOVA F-test for feature selection

`f_regression` calculates an F-statistic for each processed feature against the numerical target.

It helps identify features with stronger individual linear relationships.

It is a univariate method, so it does not identify complex interactions between features.

In [ ]:
selected_pipeline = Pipeline(
    steps=[
        ('preprocess', make_transformed_preprocessor()),
        (
            'select',
            SelectPercentile(
                score_func=f_regression,
                percentile=60,
            ),
        ),
        ('model', Ridge(alpha=1.0)),
    ]
)

selected_result = evaluate_pipeline(
    'Yeo–Johnson with F-test selection',
    selected_pipeline,
)

all_results = (
    pd.DataFrame(
        initial_results + [selected_result]
    )
    .set_index('pipeline')
)

display(all_results.round(4))

## Step 13: Inspect the selected features

A larger F-statistic indicates stronger individual linear evidence of association with engagement score.

In [ ]:
fitted_preprocessor = selected_pipeline.named_steps['preprocess']
fitted_selector = selected_pipeline.named_steps['select']

processed_feature_names = (
    fitted_preprocessor.get_feature_names_out()
)

feature_selection_audit = pd.DataFrame(
    {
        'processed_feature': processed_feature_names,
        'f_statistic': fitted_selector.scores_,
        'p_value': fitted_selector.pvalues_,
        'selected': fitted_selector.get_support(),
    }
)

feature_selection_audit['interpretation'] = np.where(
    feature_selection_audit['p_value'] < ALPHA,
    'Evidence of individual association',
    'Insufficient individual evidence',
)

feature_selection_audit = feature_selection_audit.sort_values(
    ['selected', 'f_statistic'],
    ascending=[False, False],
)

display(feature_selection_audit)

In [ ]:
selected_feature_names = processed_feature_names[
    fitted_selector.get_support()
]

print('Processed feature count:', len(processed_feature_names))
print('Selected feature count:', len(selected_feature_names))

print('\nSelected features:')

for feature in selected_feature_names:
    print('-', feature)

## Connect the workflow

| Tool | Practical decision |
|---|---|
| Skew and kurtosis | Which numerical features need investigation? |
| Q–Q plot | Where does the feature depart from normality? |
| Shapiro–Wilk | Is there statistical evidence against normality? |
| K–S diagnostic | How different is the standardised sample from a normal distribution? |
| Yeo–Johnson | Does another representation improve shape or modelling? |
| One-way ANOVA | Does a categorical feature separate target means? |
| `f_regression` | Which processed features have stronger individual relationships? |
| Test metrics | Does the complete pipeline perform better on unseen data? |

# Final learner recommendation

Recommend one of the three pipelines:

1. baseline imputation and scaling
2. Yeo–Johnson transformation
3. Yeo–Johnson with F-test feature selection

Use evidence from:

- skew and kurtosis
- Q–Q plots
- Shapiro–Wilk and K–S
- one-way ANOVA
- feature-selection scores
- test-set model performance
- interpretability and maintenance

In [ ]:
recommendation_template = pd.DataFrame(
    {
        'decision_area': [
            'Recommended pipeline',
            'Normality evidence',
            'Transformation evidence',
            'ANOVA evidence',
            'Feature-selection evidence',
            'Test-performance evidence',
            'Risk or limitation',
            'Next validation step',
        ],
        'learner_response': [
            '',
            '',
            '',
            '',
            '',
            '',
            '',
            '',
        ],
    }
)

display(recommendation_template)

# Exit check

1. What does a Q–Q plot show?
2. Does a Shapiro p-value above 0.05 prove normality?
3. Why is the K–S result in this notebook approximate?
4. Does a non-normal feature have to be removed?
5. What question does one-way ANOVA answer?
6. Why is a significant ANOVA result not a final feature-selection decision?
7. What does `f_regression` calculate?
8. Why must feature selection be inside the pipeline?
9. Which evidence should determine the final pipeline?

<details>
<summary><strong>Exit-check answers</strong></summary>

1. It compares observed quantiles with theoretical normal quantiles.
2. No. It means there is insufficient evidence to reject the null hypothesis.
3. The mean and standard deviation were estimated from the same sample.
4. No. Normality is not a universal machine learning input requirement.
5. Whether the numerical target has different means across categories.
6. Statistical evidence does not guarantee improved prediction on unseen data.
7. An individual linear F-statistic and p-value for each processed feature.
8. To prevent test data from influencing feature-selection scores.
9. Diagnostic evidence, feature relevance, business context and unseen-data performance together.

</details>

# Notebook integrity check

In [ ]:
assert not performance.empty
assert not support.empty
assert TARGET_COLUMN not in X_train.columns
assert X_train.columns.tolist() == X_test.columns.tolist()
assert pd.api.types.is_numeric_dtype(y_train)
assert len(processed_feature_names) == len(fitted_selector.scores_)
assert len(selected_feature_names) > 0
assert all_results['mean_absolute_error'].ge(0).all()
assert all_results['root_mean_squared_error'].ge(0).all()
assert all_results['r_squared'].notna().all()

print('All notebook integrity checks passed.')

## Final takeaway

Do not use one statistic to make every feature decision.

Use:

- distribution diagnostics to decide whether transformation is worth testing
- one-way ANOVA to screen categorical features
- F-test selection to rank processed features
- pipelines to prevent leakage
- unseen-data metrics to compare complete workflows
- professional judgement to balance accuracy, clarity and maintainability

The aim is not to force every feature into a normal distribution.

The aim is to build an explainable, reproducible and useful machine learning pipeline.